## Установка зависимостей/ядра
Без этого проект может не запуститься

1. `cd ~/old_home/querulus-main`
2. `python3 -m venv .venv`
3. `source .venv/bin/activate`
4. `python -m pip install -U pip setuptools wheel`
5. `pip install outboxml -e . ipykernel pandas numpy pyarrow scikit-learn catboost optuna matplotlib seaborn plotly kaleido environs pymssql openpyxl`
6. `python -m ipykernel install --user --name=querulus --display-name="Python (querulus)"`


# OutBoxML: модель 2 (TARGET_FREQ / TARGET_SEV)

Конфигурации DSM — ``config_parity.json`` / ``config_prod.json`` (CF+RG в каждом; пишет collect). Признаки и HPO — из артефактов `train_loop_new`. Fit — через AutoMLManager (`retro=False`, `hp_tune=False`). В прод идут сырые `predict_proba` / `predict` (без калибровки).

**Данные:** Hive `models.querulus_train_dataset` (партиции `model_version`/`data_date`/`dataset_version`) или кэш `data/processed/querulus_train_dataset.parquet`. Модель: semver `2.0.0`, имена CF/RG стабильные (`querulus_cf` / `querulus_rg`).

**Периоды** (см. таблицу `periods` после загрузки конфигов)
- **Train (parity):** период `parity_train`.
- **Test:** весь контрольный период test.
- **Test_prod:** 15% самых поздних дат Test (`prod_cutoff`); средние 15% (τ-cal) в Test_prod **не входят**.

**Пороги τ frequency:** parity — Val в `collect` (C3); prod — τ-cal в `collect` (C3b). В `example` только загрузка из `train_loop_new/`.

**Prod-refit:** обучение на train ∪ 70% Test; τ prod на τ-cal; оценка на Test_prod (15% свежего test).

После prod-refit: FactorsPlot, cohort, email, MLflow (prod), экспорт pickle/parquet/meta. Только prod без сравнения — см. `example_final.ipynb`.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
OUTBOXML_ROOT = PROJECT_ROOT.parent.parent
for _p in (SRC, OUTBOXML_ROOT, PROJECT_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
print("PROJECT_ROOT", PROJECT_ROOT)
print("OUTBOXML_ROOT", OUTBOXML_ROOT)


In [ ]:
import logging
import warnings

import pandas as pd
from IPython.display import display

from configs import config as querulus_outboxml_config
from querulus.training.example_pipeline import (
    export_prod_service_artifacts,
    fit_parity_models,
    fit_prod_models,
    load_example_dataset,
    load_example_thresholds,
    resolve_example_paths,
    run_parity_cross_test_metrics,
    run_prod_metrics,
    run_prod_plots_and_email,
    run_test_fin_effect,
    run_test_prod_fin_effect,
)

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.2f}".format
logging.basicConfig(level=logging.INFO, format="%(message)s")


In [ ]:
USE_SYNTHETIC = False
from querulus.naming import DEFAULT_HIVE_TABLE, MODEL_VERSION as MODEL_SEMVER
HIVE_TABLE = DEFAULT_HIVE_TABLE
MODEL_VERSION = MODEL_SEMVER  # fixed semver 2.0.0

paths = resolve_example_paths(PROJECT_ROOT, use_synthetic=USE_SYNTHETIC)
print("paths", paths)


In [ ]:
bundle = load_example_dataset(
    paths,
    hive_table=HIVE_TABLE,
    model_version=MODEL_VERSION,
)
df = bundle.df
built = bundle.built
periods = bundle.periods
CF_NAME = bundle.cf_name
RG_NAME = bundle.rg_name
MODEL_VERSION = bundle.model_version
DATASET_SOURCE = bundle.dataset_source
DATASET_PATH = bundle.dataset_path

print("df.shape", df.shape, "| DATASET_SOURCE", DATASET_SOURCE)
print("loaded configs", built["parity_path"].name, built["prod_path"].name, "←", built["configs_dir"])
display(periods["table"])
print("отсечка Test_prod (prod_cutoff):", periods["prod_cutoff"])


## Parity-модели (DSM)

Обучение на parity train (см. `periods`). Модели не идут в прод; сравнение с блоком C3 `collect` — на Test.  
Порог τ frequency загружается из артефактов collect (`train_loop_new/`), в example не подбирается.


In [ ]:
_tlr = globals().get("train_loop_result")
_collect_training = getattr(_tlr, "training", None) if _tlr is not None else None

thresholds = load_example_thresholds(PROJECT_ROOT, collect_training=_collect_training)
thr_collect = thresholds.parity

models = fit_parity_models(
    bundle,
    external_config=querulus_outboxml_config,
    threshold=thr_collect,
)
dsm_cf, dsm_rg = models.dsm_cf, models.dsm_rg


## Метрики parity: Train, Test, Test_prod

Одна обученная модель; колонки — Train (DSM train), Test (весь контрольный период), Test_prod (15% поздних дат Test, без τ-cal).  
Frequency: метрики при τ из collect. Severity: строки с `TARGET_SEV > 0` (фильтр обучения RG).

In [ ]:
run_parity_cross_test_metrics(models, bundle, threshold=thr_collect)


## Финансовый эффект на Test

Предсказания на Test; τ фиксирован (из collect). Опционально — сравнение с `fin_effect_b` из `collect` (если ядро уже выполняло блок C3).


In [ ]:
fe_parity, html_path = run_test_fin_effect(
    models,
    bundle,
    paths,
    threshold=thr_collect,
    fin_effect_collect=globals().get("fin_effect_b"),
    fin_effect_config_collect=globals().get("FIN_EFFECT_CONFIG_B"),
)


## Prod-refit и финэффект на Test_prod

Обучение prod на train ∪ 70% Test. На Test_prod (15% свежего test, τ-cal исключён) — финэффект parity и prod при **своих** τ из collect:

| Модель | Train | τ |
|--------|-------|---|
| parity | `parity_train_period` | `thr_collect` (Val) |
| prod | `prod_train_period` (train + 70% test) | `thr_prod` (τ-cal) |


In [ ]:
thr_prod = thresholds.prod
models = fit_prod_models(
    bundle,
    external_config=querulus_outboxml_config,
    threshold=thr_prod,
    parity=models,
)
dsm_cf_prod = models.dsm_cf_prod
dsm_rg_prod = models.dsm_rg_prod

run_prod_metrics(models, bundle, threshold=thr_prod)

test_prod = run_test_prod_fin_effect(models, bundle, thresholds=thresholds)
fe_prod = test_prod.prod
fe_test_prod_compare = test_prod.compare_table


## FactorsPlot, cohort и QuerulusEMailDSResult (prod)


In [ ]:
run_prod_plots_and_email(
    models,
    bundle,
    external_config=querulus_outboxml_config,
)


## Экспорт артефактов для сервиса (prod)

Запись pickle frequency/severity и ансамбля, файла границ DQ, parquet `df_for_service` (исходный датасет с колонками `preds_cf` / `preds_rg`) и JSON с метаданными (`querulus_meta_*.json`), включая `best_threshold` = `thr_prod` (из collect C3b).


In [ ]:
export_result = export_prod_service_artifacts(
    models,
    bundle,
    paths,
    thresholds=thresholds,
)
meta = export_result.meta
